# VisualSeek — Kaggle T4 ×2 → Hugging Face embedding storage

Notebook dành cho dataset **Vietnamese Culture VQA** trong project VisualSeek.
Luồng tương tự notebook mẫu: cấu hình → chuẩn bị code/data → chạy GPU → kiểm tra → đóng gói → upload.
Dùng **SigLIP2 cùng revision/preprocessing với backend**, không huấn luyện lại model.

**Trước khi Run All:**
1. Trong Kaggle chọn **Settings / Session options → Accelerator → GPU T4 x2**, bật **Internet**.
   Code kiểm tra có 2 GPU; notebook không thể tự cấp GPU nếu tài khoản chưa được Kaggle cấp.
2. Add Input: [dataset ảnh nguồn](https://www.kaggle.com/datasets/blackcubo/vietnamese-culture-vqa-dataset)
   (cùng version/file bytes với local), hoặc dataset ảnh riêng giữ cây `images/<category>/...`.
3. Add Input chứa **`culture-kaggle-code.zip` mới** và **`culture-selection.zip`** của project.
   Hai ZIP đã có ở `data/exports/`; có thể đưa vào cùng một Kaggle Dataset riêng.
4. Thêm **Kaggle Secret `HF_TOKEN`**, bật quyền truy cập cho notebook. Token có quyền write vào
   **`Quanganh6905/Embedding_for_searchengine`**; nếu repo chưa có, token cần quyền tạo Dataset repo.
   Không dán token vào code/output/chat.

Tạo lại hai bundle trên máy local khi code hoặc manifest thay đổi:
```bash
python3 scripts/prepare_kaggle_bundle.py
python3 scripts/prepare_kaggle_selection.py
```
Manifest local là allowlist. Bản hiện tại có **15.423 asset**, nhưng notebook luôn đọc số thực tế từ
selection, không hard-code tổng số ảnh. Không đưa VQA questions/answers vào embedding hoặc artifact.

Mỗi GPU chạy một process riêng, nhận một nửa allowlist và dùng FP16 inference. Batch size là **theo GPU**;
hai GPU có bộ nhớ riêng, không gộp thành một vùng VRAM. Image vectors và metadata vectors đều L2/float32.


## 1. Cấu hình
Để chạy thử, đổi `LIMIT_PER_CATEGORY = 20` và đặt `RUN_NAME` riêng.
`0` chạy toàn bộ manifest local. Khi đổi tập ảnh/model/số GPU, dùng tên run mới.


In [ ]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
WORK_ROOT.mkdir(parents=True, exist_ok=True)

HF_REPO_ID = 'Quanganh6905/Embedding_for_searchengine'
HF_SECRET_NAME = 'HF_TOKEN'
HF_PRIVATE = True  # Chỉ áp dụng nếu tạo repo mới; không đổi visibility repo đã tồn tại.
RUN_NAME = 'siglip2-base-224-t4x2-full-v1'
GPU_IDS = [0, 1]
BATCH_SIZE_PER_GPU = 16  # Global: tối đa 32 ảnh/lượt khi cả hai worker có batch đầy.
LIMIT_PER_CATEGORY = 0
WITH_METADATA = True

CODE_SOURCE = None  # Hoặc đường dẫn ZIP/thư mục code cụ thể trong /kaggle/input.
SELECTION_SOURCE = None  # Hoặc đường dẫn ZIP/thư mục selection cụ thể.
IMAGE_ROOT = None  # Thư mục CHA của images/; tự tìm nếu None.
OUTPUT = WORK_ROOT / RUN_NAME
CACHE_DIR = WORK_ROOT / 'model-cache'
HF_SUBDIR = f'artifacts/{RUN_NAME}'

if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9._-]*', RUN_NAME):
    raise ValueError('RUN_NAME chỉ gồm chữ, số, dấu chấm, gạch ngang hoặc gạch dưới')
if BATCH_SIZE_PER_GPU < 1 or LIMIT_PER_CATEGORY < 0 or GPU_IDS != [0, 1]:
    raise ValueError('Cần batch > 0, limit >= 0, GPU_IDS = [0, 1] cho notebook T4x2')

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')
print('HF Dataset:', HF_REPO_ID, '| Run:', RUN_NAME)


## 2. Nạp code bundle và cài dependencies
Giữ nguyên PyTorch/CUDA có sẵn của Kaggle. Hỗ trợ ZIP hoặc bundle đã được Kaggle giải nén.
Nếu có nhiều bản bundle, đặt `CODE_SOURCE` / `SELECTION_SOURCE` đúng đường dẫn được in ra.


In [ ]:
def find_bundle(zip_name, marker, required_files, explicit=None):
    if explicit is not None:
        source = Path(explicit)
        if not source.exists():
            raise FileNotFoundError(f'Configured source does not exist: {source}')
        return source
    archives = [p for p in INPUT_ROOT.rglob(zip_name) if p.is_file()]
    directories = set()
    marker = Path(marker)
    for file in INPUT_ROOT.rglob(marker.name):
        root = file.parents[len(marker.parts) - 1]
        if all((root / name).is_file() for name in required_files):
            directories.add(root)
    candidates = sorted(set(archives) | directories)
    print(f'Candidates for {zip_name}:', [str(p) for p in candidates])
    if not candidates:
        print('Attached inputs:', [str(p) for p in INPUT_ROOT.iterdir()])
        raise FileNotFoundError(
            f'Attach {zip_name} as a Notebook Input, or attach its extracted files. '
            'The image dataset alone is not enough. If the code Input is an old '
            'incomplete bundle, regenerate it and update the Input version.'
        )
    if len(candidates) > 1:
        raise RuntimeError('Multiple bundles found. Set the explicit source to one printed path.')
    return candidates[0]

def materialize_bundle(source, destination, required_files):
    if source.is_dir():
        root = source  # Kaggle Input có thể đọc trực tiếp, không cần ghi vào đó.
    else:
        root = Path(destination)
        root.mkdir(parents=True, exist_ok=True)
        with ZipFile(source) as archive:
            missing = sorted(set(required_files) - set(archive.namelist()))
            if missing:
                raise FileNotFoundError(
                    f'Incomplete bundle {source}: {missing}. Regenerate and update the Input.'
                )
            # Giữ đủ module trong ZIP, kể cả module mới chưa nằm trong required_files.
            for member in archive.infolist():
                path = PurePosixPath(member.filename)
                if (path.is_absolute() or '..' in path.parts or '\\' in member.filename
                        or not (root / member.filename).resolve().is_relative_to(root.resolve())):
                    raise ValueError(f'Unsafe bundle path: {member.filename}')
            archive.extractall(root)
    missing = [name for name in required_files if not (root / name).is_file()]
    if missing:
        raise FileNotFoundError(
            f'Incomplete bundle directory {root}: {missing}. Regenerate and update the Input.'
        )
    return root

code_files = [
    'scripts/kaggle_embed.py', 'scripts/_bootstrap.py', 'scripts/kaggle-requirements.txt',
    'scripts/enrich_embeddings.py', 'scripts/kaggle_embed_multi_gpu.py',
    'packages/embedding/src/visualseek_embedding/__init__.py',
    'packages/embedding/src/visualseek_embedding/contracts.py',
    'packages/embedding/src/visualseek_embedding/siglip2.py',
    'packages/embedding/src/visualseek_embedding/metadata.py',
    'packages/retrieval/src/visualseek_retrieval/__init__.py',
    'packages/retrieval/src/visualseek_retrieval/contracts.py',
    'packages/retrieval/src/visualseek_retrieval/artifact.py',
    'packages/preprocessing/src/visualseek_preprocessing/__init__.py',
    'packages/preprocessing/src/visualseek_preprocessing/contracts.py',
    'packages/preprocessing/src/visualseek_preprocessing/culture_dataset.py',
]

source = find_bundle('culture-kaggle-code.zip', 'scripts/kaggle_embed.py', code_files, CODE_SOURCE)
code_root = materialize_bundle(source, WORK_ROOT / 'visualseek-code', code_files)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(code_root / 'scripts/kaggle-requirements.txt'),
    'huggingface_hub==0.36.0',
], check=True)
subprocess.run([
    sys.executable, str(code_root / 'scripts/kaggle_embed_multi_gpu.py'), '--help',
], check=True)
print('Code ready:', code_root)


## 3. Kiểm tra GPU và đăng nhập Hugging Face
Dừng sớm nếu thiếu GPU hoặc token. Đọc secret trong bộ nhớ, không lưu token vào notebook hay bundle.
Repo được tạo ở bước upload sau khi artifact đã qua kiểm tra.


In [ ]:
import torch
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError('Cần 2 GPU: chọn Settings → Accelerator → GPU T4 x2, rồi khởi động lại session')
for gpu in GPU_IDS:
    info = torch.cuda.get_device_properties(gpu)
    print(f'GPU {gpu}: {info.name}, VRAM={info.total_memory / 2**30:.1f} GiB')
    if 'T4' not in info.name:
        raise RuntimeError('Notebook này cấu hình cho T4x2; hãy chọn đúng accelerator trên Kaggle')
print('Torch:', torch.__version__, '| CUDA:', torch.version.cuda)
subprocess.run(['nvidia-smi'], check=True)

hf_token = UserSecretsClient().get_secret(HF_SECRET_NAME)
if not hf_token:
    raise RuntimeError(f'Kaggle Secret {HF_SECRET_NAME} chưa có giá trị')
api = HfApi(token=hf_token)
del hf_token
identity = api.whoami()
print('Hugging Face user:', identity['name'], '| Dataset đích:', HF_REPO_ID)
# whoami xác thực token; quyền ghi vào repo được HF kiểm tra khi create/upload.


## 4. Nạp đúng manifest local và tìm ảnh
Kiểm tra hash bundle selection, danh sách asset và đường dẫn ảnh. Script ở bước kế tiếp
kiểm tra SHA-256 từng ảnh trước khi chạy model; không tự thay ảnh thiếu bằng ảnh khác.
Nếu chỉ attach ZIP ảnh gốc, hãy giải nén vào `/kaggle/working` và đặt `IMAGE_ROOT` tương ứng.


In [ ]:
selection_files = ['culture-assets.jsonl', 'selection.json']
selection_source = find_bundle(
    'culture-selection.zip', 'culture-assets.jsonl', selection_files, SELECTION_SOURCE
)
selection_dir = materialize_bundle(
    selection_source, WORK_ROOT / 'local-selection', selection_files
)
local_manifest = selection_dir / 'culture-assets.jsonl'
selection = json.loads((selection_dir / 'selection.json').read_text(encoding='utf-8'))
if hashlib.sha256(local_manifest.read_bytes()).hexdigest() != selection['manifest_sha256']:
    raise ValueError('Selection manifest checksum mismatch')
local_rows = [json.loads(line) for line in local_manifest.read_text(encoding='utf-8').splitlines()
              if line.strip()]
expected_ids = {row['asset_id'] for row in local_rows}
if len(expected_ids) != len(local_rows) or len(local_rows) != selection['count']:
    raise ValueError('Selection count/unique asset IDs mismatch')
ids_hash = hashlib.sha256('\n'.join(sorted(expected_ids)).encode()).hexdigest()
if ids_hash != selection['asset_ids_sha256']:
    raise ValueError('Selection asset IDs checksum mismatch')

sys.path.insert(0, str(code_root / 'scripts'))
import _bootstrap  # Nạp các package project từ bundle, không cài editable workspace.
from visualseek_preprocessing.culture_dataset import select_records
selected_rows = select_records(local_rows, LIMIT_PER_CATEGORY or None)
selected_ids = {row['asset_id'] for row in selected_rows}
print('Local allowlist:', selection['count'], selection['categories'])
print('Run sẽ embed:', len(selected_ids), 'asset')


In [ ]:
def find_image_root(rows, input_root, explicit=None):
    relatives = [PurePosixPath(row['image_path']) for row in rows]
    if not relatives or any(path.is_absolute() or '..' in path.parts
                            or not path.parts or path.parts[0] != 'images'
                            for path in relatives):
        raise ValueError('The local manifest must contain safe images/... paths')
    candidates = ([Path(explicit)] if explicit is not None else sorted({
        directory.parent.resolve() for directory in input_root.rglob('images')
        if directory.is_dir()
    }))
    print('Image root candidates:', [str(root) for root in candidates])
    if not candidates:
        print('Attached inputs:', [str(path) for path in input_root.iterdir()])
        raise FileNotFoundError(
            'No extracted images/ folder found. Attach the image dataset, not only '
            'code/selection or annotation files. If images are still zipped, extract '
            'them to /kaggle/working and set IMAGE_ROOT to the parent of images/.'
        )
    matches = []
    for root in candidates:
        missing = []
        present = 0
        for relative in relatives:
            if (root / str(relative)).is_file():
                present += 1
            elif len(missing) < 3:
                missing.append(str(relative))
        print(f'{root}: {present}/{len(relatives)} manifest images found; missing examples={missing}')
        if present == len(relatives):
            matches.append(root)
    if not matches:
        raise FileNotFoundError(
            'No candidate contains the complete local image selection. Check the '
            'dataset version, directory layout and IMAGE_ROOT. Do not remove missing '
            'rows just to make the check pass.'
        )
    if len(matches) > 1:
        raise RuntimeError('Multiple complete image roots found. Set IMAGE_ROOT to one printed path.')
    return matches[0]

image_root = find_image_root(local_rows, INPUT_ROOT, IMAGE_ROOT)
print('Image root:', image_root)


## 5. Chạy đồng thời 2 GPU, checkpoint rồi gộp artifact
Hai subprocess cùng chạy với `CUDA_VISIBLE_DEVICES=0` và `CUDA_VISIBLE_DEVICES=1`.
Mỗi process dùng `cuda:0` trong không gian GPU riêng; không cần DDP/NCCL cho inference độc lập.
Giữ nguyên model, revision, EXIF/RGB, metadata chunking và mapping asset của backend.

Log `[GPU 0]` / `[GPU 1]` hiển thị tiến độ; log đầy đủ và checkpoint nằm trong
`OUTPUT/workers/rank-*/`. Giai đoạn kiểm tra checksum/decode ảnh chạy CPU/ổ đĩa.
Nếu lỗi, cell dừng và giữ checkpoint; chỉ gộp khi **cả hai worker thành công**.


In [ ]:
command = [
    sys.executable, '-u', str(code_root / 'scripts/kaggle_embed_multi_gpu.py'),
    '--input-root', str(image_root), '--manifest', str(local_manifest),
    '--output', str(OUTPUT), '--cache-dir', str(CACHE_DIR),
    '--gpu-ids', *map(str, GPU_IDS),
    '--batch-size', str(BATCH_SIZE_PER_GPU),
    '--limit-per-category', str(LIMIT_PER_CATEGORY),
]
if WITH_METADATA:
    command.append('--with-metadata')
subprocess.run(command, check=True)


## 6. Kiểm tra artifact cuối cùng
Kiểm tra checksum, shape, L2, count, model revision, đủ image/metadata vectors và tập asset đúng local.
ZIP chứa `vectors.npy`, `metadata_vectors.npy`, `assets.jsonl`, `manifest.json`, `smoke_report.json`;
không chứa ảnh gốc hoặc model weights. Có thể import trực tiếp bằng script hiện có của project.


In [ ]:
from visualseek_embedding.siglip2 import MODEL_ID, MODEL_REVISION
from visualseek_retrieval.artifact import ArtifactIndex, sha256_file

index = ArtifactIndex(OUTPUT)
manifest = index.manifest
report = json.loads((OUTPUT / 'smoke_report.json').read_text(encoding='utf-8'))
exported_ids = {row['asset_id'] for row in index.assets}
if not manifest['complete'] or manifest['count'] != manifest['requested']:
    raise RuntimeError('Artifact chưa đầy đủ')
if report['failures'] != 0 or exported_ids != selected_ids:
    raise RuntimeError('Asset export không khớp allowlist đã chọn')
if manifest['model_id'] != MODEL_ID or manifest['model_revision'] != MODEL_REVISION:
    raise RuntimeError('Model/revision không khớp backend')
if WITH_METADATA and index.metadata_vectors is None:
    raise RuntimeError('Thiếu metadata vectors cho contextual/hybrid search')
if LIMIT_PER_CATEGORY == 0 and exported_ids != expected_ids:
    raise RuntimeError('Full run phải khớp toàn bộ manifest local')
archive = OUTPUT / 'culture-embeddings.zip'
with ZipFile(archive) as bundle:
    if bundle.testzip() is not None:
        raise RuntimeError('ZIP lỗi CRC')
shutil.copyfile(selection_dir / 'selection.json', OUTPUT / 'selection.json')
print('Image vectors:', index.vectors.shape, index.vectors.dtype)
print('Metadata vectors:', None if index.metadata_vectors is None else index.metadata_vectors.shape)
print('Categories:', index.counts)
print('ZIP:', archive, f'({archive.stat().st_size / 2**20:.1f} MiB)')
print('ZIP SHA256:', sha256_file(archive))
print('Worker smoke tests kiểm tra pipeline trên từng nửa corpus; chưa đo chất lượng retrieval.')


## 7. Upload lên Hugging Face Dataset
Repo đích **`Quanganh6905/Embedding_for_searchengine`**, thư mục `artifacts/<RUN_NAME>/`.
Tạo repo private nếu chưa có; repo tồn tại giữ nguyên visibility. Upload theo allowlist:
ZIP embedding + manifest + selection + báo cáo + README, không upload thư mục cache/checkpoint.
Sau upload lưu commit SHA để tải lại đúng phiên bản. Có thể chạy lại riêng cell này khi lỗi mạng.


In [ ]:
# Kiểm tra lại trước mỗi lần upload riêng cell này.
validated = ArtifactIndex(OUTPUT)
if (not validated.manifest['complete']
        or {row['asset_id'] for row in validated.assets} != selected_ids):
    raise RuntimeError('Artifact không còn khớp selection; chạy lại bước kiểm tra')
api.create_repo(repo_id=HF_REPO_ID, repo_type='dataset', private=HF_PRIVATE, exist_ok=True)
readme = f'''# VisualSeek embedding artifact: {RUN_NAME}

- Model: `{MODEL_ID}@{MODEL_REVISION}`
- Assets: {manifest['count']}; dimension: {manifest['dimension']}; vectors: float32, L2.
- Image vectors from pixels only; metadata vectors from keyword/tags/description, without VQA labels.
- Source: https://www.kaggle.com/datasets/blackcubo/vietnamese-culture-vqa-dataset
- Selection: exact project-local allowlist; limit per category: {LIMIT_PER_CATEGORY} (0 = full).
- Generated by two isolated Kaggle T4 GPU processes.
- `culture-embeddings.zip` contains vectors + aligned asset metadata + checksum manifest.
- Original images and model weights are not included.
- Worker smoke queries search worker subsets; retrieval relevance is not measured.

Download this ZIP with `hf_hub_download(repo_type="dataset", revision="<commit SHA>")`,
then run `python3 scripts/import_embeddings.py <ZIP> --output data/embeddings/<new-version>`.
'''
(OUTPUT / 'README.md').write_text(readme, encoding='utf-8')
upload_names = [
    'culture-embeddings.zip', 'manifest.json', 'selection.json',
    'smoke_report.json', 'failures.jsonl', 'multi-gpu-run.json', 'README.md',
]
commit = api.upload_folder(
    repo_id=HF_REPO_ID, repo_type='dataset',
    folder_path=str(OUTPUT), path_in_repo=HF_SUBDIR,
    allow_patterns=upload_names,
    commit_message=f'Add VisualSeek SigLIP2 embedding artifact: {RUN_NAME}',
)
ARTIFACT_COMMIT = commit.oid
remote_files = set(api.list_repo_files(
    repo_id=HF_REPO_ID, repo_type='dataset', revision=ARTIFACT_COMMIT
))
missing = {f'{HF_SUBDIR}/{name}' for name in upload_names} - remote_files
if missing:
    raise RuntimeError(f'Remote commit thiếu file: {sorted(missing)}')
receipt = {
    'repo_id': HF_REPO_ID, 'repo_type': 'dataset', 'revision': ARTIFACT_COMMIT,
    'filename': f'{HF_SUBDIR}/culture-embeddings.zip',
    'sha256': sha256_file(archive), 'count': manifest['count'],
}
receipt_path = WORK_ROOT / f'{RUN_NAME}-upload-receipt.json'
receipt_path.write_text(json.dumps(receipt, indent=2) + '\n', encoding='utf-8')
print('Uploaded:', f'https://huggingface.co/datasets/{HF_REPO_ID}/tree/{ARTIFACT_COMMIT}/{HF_SUBDIR}')
print('Commit SHA:', ARTIFACT_COMMIT)
print('Giữ receipt để tải đúng phiên bản:', receipt_path)
print(json.dumps(receipt, indent=2))


## 8. Tải về và import trên máy local (không cần embed lại corpus)
Các lệnh sau **chạy trên máy local**, không phải một cell Python của Kaggle.
Với Dataset private, đăng nhập HF trên máy local hoặc đặt `HF_TOKEN` bằng cách an toàn.
Copy các giá trị từ upload receipt ở cell trên:

```python
from huggingface_hub import hf_hub_download

archive = hf_hub_download(
    repo_id='Quanganh6905/Embedding_for_searchengine',
    repo_type='dataset',
    filename='artifacts/siglip2-base-224-t4x2-full-v1/culture-embeddings.zip',
    revision='COPY_COMMIT_SHA_FROM_UPLOAD_RECEIPT',
)
print(archive)
```
```bash
python3 scripts/import_embeddings.py /path/to/downloaded/culture-embeddings.zip \
  --output data/embeddings/culture-siglip2-t4x2-v1
```
`import_embeddings.py` xác minh checksum và schema trước khi cài artifact vào thư mục mới.
Cấu hình backend theo `docs/kaggle-embedding-rag.md` / `docs/contextual-image-search.md`.
Backend vẫn dùng cùng ảnh local và encode query bằng đúng SigLIP2 revision.

**Tiếp tục và xử lý lỗi:**
- Thiếu 2 GPU: chọn T4x2 trong Kaggle UI; metadata notebook không tự cấp thêm GPU.
- OOM ở bước image embedding: giảm batch còn 8/4 rồi chạy lại; shard ảnh hoàn tất được tái sử dụng.
  Nếu checkpoint metadata đã được tạo, giữ batch size cũ hoặc dùng `RUN_NAME` mới khi đổi batch.
- Lỗi ảnh/checksum: kiểm tra version dataset/cây thư mục và manifest, không bỏ hàng để chạy cho qua.
- Output có corpus/model/GPU khác: dùng `RUN_NAME` mới.
- Resume cùng session: chạy lại cell 5. Qua session mới, attach output đã Save Version,
  copy **toàn bộ thư mục run gồm workers/checkpoint** từ Input về `OUTPUT`, rồi chạy lại.
  ZIP final không chứa checkpoint. Không đảm bảo giữ được tiến độ chưa lưu khi Kaggle ngắt session.
- HF 401/403: cấp token write (và quyền tạo repo nếu cần), bật quyền Secret, chạy lại cell 7;
  không cần tạo embedding lại.

Tài liệu: [Kaggle GPU settings](https://www.kaggle.com/docs/notebooks),
[PyTorch GPU isolation](https://docs.pytorch.org/docs/stable/notes/cuda.html#cuda-environment-variables),
[HF upload API 0.36](https://huggingface.co/docs/huggingface_hub/v0.36.0/en/package_reference/hf_api#huggingface_hub.HfApi.upload_folder).
